# finetune_run — huấn luyện + đo, MỘT phiên GPU

Gộp hai việc để không tốn hai lần khởi động phiên:

1. Fine-tune `AITeamVN/Vietnamese_Reranker` trên 6000 câu train
2. Đo ngay trên dev300, so với mốc hiện tại **0.8883**

**Không** chấm đề thi ở đây. Nếu bước 2 thắng mốc thì mới chạy `finalAnswer_run.ipynb`
với `RERANKER_MODEL` trỏ vào `ft_model` — tách ra để không phí 5 tiếng khi fine-tune hỏng.

Ước ~3 tiếng. Bật **GPU T4 x2**, dùng **Save & Run All (Commit)**.

> Hard negative cần `bm25_ids_train_FALLBACK.json` (sinh ở máy, CPU).
> Không có file đó thì rơi về negative ngẫu nhiên — vẫn chạy, nhưng yếu hơn nhiều.

## Cài package + trỏ dataset

In [ ]:
!pip install -q sentence-transformers

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys
INPUT_DIR = "/kaggle/input/project-ir"
sys.path.append(INPUT_DIR)
!ls /kaggle/input

In [ ]:
import json, os, time
from pathlib import Path

from metrics import evaluate
from rerank import load_reranker
from rerank_from_d import score_all_from_d, blend_bm25_first
import finetune_rerank as FT

## Config

In [ ]:
BASE_MODEL   = "AITeamVN/Vietnamese_Reranker"
OUT_MODEL    = "/kaggle/working/ft_model"

TRAIN_PATH   = f"{INPUT_DIR}/train.json"
DEV_EXCLUDE  = f"{INPUT_DIR}/dev_1000_locked.json"   # bao trùm dev300 + dev150
CTX_DIR      = f"{INPUT_DIR}/selected-contexts"

# Hard negative. None -> negative ngẫu nhiên (yếu hơn nhiều).
HARD_NEG     = f"{INPUT_DIR}/bm25_ids_train_FALLBACK.json"

DEV_GOLD     = f"{INPUT_DIR}/dev_300_locked.json"
DEV_CAND     = f"{INPUT_DIR}/bm25_top100_dev300.json"

N_NEG, EPOCHS, BS, LR, MAXLEN = 4, 1, 8, 2e-5, 512   # bs=16 -> OOM tren T4 16GB (12/08)
K, N_BM25 = 5, 2

# Mốc phải vượt: AITeamVN gốc + blend n=2 trên dev300
BASELINE = 0.8883

OUTPUT_DIR = "/kaggle/working/outputs"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

## Bước 1 — Dựng dữ liệu huấn luyện

Chạy trên CPU, vài phút. Ba chốt chặn quy chế trong `finetune_rerank.py` tự kiểm:
mọi `question_id` phải thuộc train, mọi `doc_id` phải thuộc corpus, không sinh chữ mới.

Dòng `hard negative: x/y` cho biết có thật sự dùng hard negative không — nếu ra 0
thì file candidate không khớp qid, đang rơi về ngẫu nhiên.

In [ ]:
t0 = time.time()
train = json.load(open(TRAIN_PATH, encoding="utf-8"))
dev_qids = set(json.load(open(DEV_EXCLUDE, encoding="utf-8")))
corpus_ids = {f[len("context_"):-len(".json")] for f in os.listdir(CTX_DIR) if f.startswith("context_")}

cands = None
if HARD_NEG and os.path.exists(HARD_NEG):
    cands = json.load(open(HARD_NEG, encoding="utf-8"))
    print(f"hard negative: {len(cands):,} câu có candidate")
else:
    print("[!] KHÔNG có file hard negative -> dùng negative NGẪU NHIÊN (yếu hơn nhiều)")

print(f"train {len(train):,} | loại dev {len(dev_qids):,} | corpus {len(corpus_ids):,}")
pairs = FT.build_pairs(train, dev_qids, corpus_ids, CTX_DIR, cands, n_neg=N_NEG)
print(f"dựng xong trong {time.time()-t0:.0f}s")

assert not (dev_qids & {q for q in train if q not in dev_qids}), "RÒ RỈ DEV"
print("[ok] không câu dev nào lọt vào tập huấn luyện")

## Bước 2 — Huấn luyện

~1 giờ với 30.000 cặp, `max_length=512`. Loss nên giảm dần; nếu nó nhảy loạn
hoặc đứng yên thì dừng, đừng để chạy hết rồi mới biết.

In [ ]:
t0 = time.time()
FT.train_model(pairs, BASE_MODEL, OUT_MODEL,
               epochs=EPOCHS, bs=BS, lr=LR, max_length=MAXLEN, device="cuda")
print(f"huấn luyện xong trong {(time.time()-t0)/60:.0f} phút")
del pairs

## Bước 3 — Đo trên dev300

So model vừa huấn luyện với mốc **0.8883**. Bảng `n_bm25` in lại đầy đủ vì
tham số tối ưu có thể đổi sau khi fine-tune.

In [ ]:
dev = json.load(open(DEV_GOLD, encoding="utf-8"))
dev_cand = json.load(open(DEV_CAND, encoding="utf-8"))
dev_q = {q: v["question"] for q, v in dev.items()}
dev_gold = {q: v["answer"] for q, v in dev.items()}

thieu = set(dev_q) - set(dev_cand)
assert not thieu, f"{len(thieu)} câu thiếu candidate"

score_fn = load_reranker(OUT_MODEL, device="cuda")

t0 = time.time()
scores = score_all_from_d(dev_q, dev_cand, score_fn)
print(f"chấm xong trong {(time.time()-t0)/60:.0f} phút\n")

rank = lambda q: [d for d, _ in sorted(scores[q].items(), key=lambda x: -x[1]["ce"])]
bm25 = {q: [str(c["doc_id"]) for c in dev_cand[q]] for q in dev_q}
do = lambda n: evaluate(dev_gold,
                        {q: blend_bm25_first(rank(q), bm25[q], k=K, n_bm25=n) for q in dev_q},
                        k=K)["recall"]

row = {n: do(n) for n in (0, 1, 2, 3)}
for n, v in row.items():
    print(f"  n_bm25={n}   {v:.4f}" + ("   <-- cấu hình chốt" if n == N_BM25 else ""))

best_n = max(row, key=row.get)
ft = row[N_BM25]
print(f"\nfine-tune (n={N_BM25}) : {ft:.4f}")
print(f"mốc gốc   (n={N_BM25}) : {BASELINE:.4f}")
print(f"chênh lệch            : {ft - BASELINE:+.4f}")
print(f"n tốt nhất sau fine-tune: {best_n} ({row[best_n]:.4f})")
print("\n=> " + ("ĐÁNG DÙNG. Chạy finalAnswer_run với RERANKER_MODEL = ft_model."
                 if row[best_n] > BASELINE else
                 "KHÔNG ĐÁNG. Giữ model gốc, đừng chạy đề thi."))

## Bước 4 — Lưu lại

In [ ]:
json.dump(scores, open(f"{OUTPUT_DIR}/scores_dev300_ft.json", "w", encoding="utf-8"),
          ensure_ascii=False)
json.dump({"recall_by_n": row, "baseline": BASELINE, "best_n": best_n,
           "base_model": BASE_MODEL, "n_neg": N_NEG, "epochs": EPOCHS,
           "lr": LR, "max_length": MAXLEN,
           "hard_negative": bool(cands)},
          open(f"{OUTPUT_DIR}/ft_eval.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)

for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {f}  {os.path.getsize(os.path.join(OUTPUT_DIR, f)):,} bytes")
print(f"\nTẢI VỀ: outputs/ và {OUT_MODEL} (Kaggle xoá /kaggle/working sau phiên)")